In [ ]:
# ========== 第 7 周练习：SaaS 线索评分 —— 前沿模型打标 + QLoRA 微调开源模型 ==========
# 练习目标：用 GPT-4o-mini（经 OpenRouter）给招聘 JD 打 Hot/Warm/Cold 分，
# 再把结果做成 Llama 指令数据，用 4-bit QLoRA 微调，最后对比两者一致性。
# 需 Colab + GPU + Secrets(OPENROUTER_API_KEY, HF_TOKEN) + Google Drive。
# 可运行逻辑未改；下列为依赖安装。

# transformers/datasets/peft/bitsandbytes/accelerate/trl：微调与量化栈
!pip install -q transformers datasets peft bitsandbytes accelerate trl
# openai：调 OpenRouter；sklearn/pandas/numpy/matplotlib/seaborn：评估与画图
!pip install -q openai scikit-learn pandas numpy matplotlib seaborn


In [ ]:
# ========== 导入：数据处理 / API / HF 微调栈 / GPU 检查 ==========

# 标准库：路径、JSON、限速 sleep
import os
import json
import time
# 表格与数值
import pandas as pd
import numpy as np
# 可视化
import matplotlib.pyplot as plt
import seaborn as sns
# Hugging Face Hub 登录（门禁模型 / 数据集）
from huggingface_hub import login

# OpenAI 兼容客户端（后面指向 OpenRouter）
from openai import OpenAI
# 加载 HF 数据集
from datasets import load_dataset
# 分类报告与混淆矩阵（本练习主用 tier 对比，导入保留）
from sklearn.metrics import classification_report, confusion_matrix

# PyTorch + Transformers 因果语言模型与训练参数
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
# PEFT / LoRA：低秩适配器
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# TRL 的 SFTTrainer：监督微调（Supervised Fine-Tuning）
from trl import SFTTrainer

# 自检：导入成功 + 是否有 GPU
print("✓ All imports successful")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ========== Colab Secrets + Drive 挂载 + 常量/路径配置 ==========

# userdata：读 Colab Secrets；drive：挂载 Google Drive 做持久化
from google.colab import userdata, drive

# 挂载云盘：训练产物与结果 JSON 写到 Drive，断线不丢
drive.mount("/content/drive")

# 从 Secrets 取 API Key（不要写进笔记本正文）
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
HF_TOKEN           = userdata.get("HF_TOKEN")

# OpenRouter：OpenAI 兼容网关 + 前沿小模型 id
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
FRONTIER_MODEL      = "openai/gpt-4o-mini"

# 招聘 JD 数据集名与划分；SAMPLE_SIZE 控制打标样本量
DATASET_NAME  = "jacob-hugging-face/job-descriptions"
DATASET_SPLIT = "train"
SAMPLE_SIZE   = 200

# 待微调的开源 Instruct 模型（需 HF 授权）
OSS_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# 分数 → 档位（tier）区间：Hot / Warm / Cold
SCORE_TIERS = {
    "Hot":  (70, 100),
    "Warm": (40, 69),
    "Cold": (0,  39),
}

# 所有输出放 Drive，避免 Colab 本地盘被清
DRIVE_BASE          = "/content/drive/MyDrive/saas_lead_scoring"
OUTPUT_DIR          = f"{DRIVE_BASE}/outputs"
FINE_TUNED_MODEL_DIR = f"{DRIVE_BASE}/fine_tuned_model"
RESULTS_FILE        = "frontier_results.json"

# 目录不存在则创建
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINE_TUNED_MODEL_DIR, exist_ok=True)

# 缺密钥直接断言失败，避免后面静默报错
assert OPENROUTER_API_KEY, "Missing secret: OPENROUTER_API_KEY"
assert HF_TOKEN,           "Missing secret: HF_TOKEN"

print("✓ Google Drive mounted")
print("✓ Secrets loaded successfully")
print(f"✓ Frontier model  : {FRONTIER_MODEL}")
print(f"✓ OSS model       : {OSS_MODEL_NAME}")
print(f"✓ Outputs dir     : {OUTPUT_DIR}")
print(f"✓ Model dir       : {FINE_TUNED_MODEL_DIR}")


In [ ]:
# ========== 登录 HF + 加载数据集并转成 DataFrame ==========

from huggingface_hub import login

# 用 HF_TOKEN 登录；不写入 git credential
login(token=HF_TOKEN, add_to_git_credential=False)

# 按常量加载指定 split，再转 pandas 方便清洗
dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
df = pd.DataFrame(dataset)

# 看规模、列名、第一条样例
print(f"✓ Dataset loaded: {len(df)} records")
print(f"✓ Columns: {df.columns.tolist()}")
print(f"\n--- Sample Record ---")
print(df.iloc[0].to_dict())


In [ ]:
# ========== 探索：model_response 结构、空值、描述长度 ==========

# 看一条 model_response 原文，了解是否为 JSON 字符串
print("--- Sample model_response ---")
print(df["model_response"].iloc[0])
# 各列空值计数
print("\n--- Null counts ---")
print(df.isnull().sum())
# 描述长度统计（若有 description_length 列）
print(f"\n--- Description length stats ---")
print(df["description_length"].describe())


In [ ]:
# ========== 清洗：解析 JSON、去短 JD、抽样 ==========

def parse_model_response(raw):
    """把 model_response 字符串解析成 dict；失败返回 None。"""
    try:
        return json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None

# 逐行 apply 解析，得到 parsed_response 列
df["parsed_response"] = df["model_response"].apply(parse_model_response)

# 丢掉解析失败的行，并重置索引
df = df[df["parsed_response"].notnull()].reset_index(drop=True)

# 只保留后续打分需要的列
df = df[["company_name", "position_title", "job_description", "parsed_response"]]

# 过滤过短 JD（可能是垃圾/残缺）
df = df[df["job_description"].str.len() >= 100].reset_index(drop=True)

# 固定 random_state，保证可复现抽样
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)

print(f"✓ Clean dataset  : {len(df)} records")
print(f"✓ Sample size    : {len(df_sample)} records")
print(f"\n--- Sample Row ---")
print(f"Company   : {df_sample['company_name'].iloc[0]}")
print(f"Position  : {df_sample['position_title'].iloc[0]}")
print(f"JD length : {len(df_sample['job_description'].iloc[0])} chars")
print(f"Parsed    : {json.dumps(df_sample['parsed_response'].iloc[0], indent=2)}")


In [ ]:
# ========== 数据质量过滤：parsed 里至少 2 个有效字段 ==========

def is_valid_response(parsed):
    """检查解析结果是否「够有信息量」。"""
    if not parsed:
        return False
    valid_fields = []
    for v in parsed.values():
        # 列表 → 拼成字符串；字符串直接用；其它类型跳过
        if isinstance(v, list):
            text = " ".join(v).strip()
        elif isinstance(v, str):
            text = v.strip()
        else:
            continue
        # 排除占位符 N/A / NONE / NULL / 空串
        if text.upper() not in ("N/A", "NONE", "NULL", ""):
            valid_fields.append(text)
    # 至少两个有效字段才算过关
    return len(valid_fields) >= 2

# 主表打标 is_valid，再筛出干净子集
df["is_valid"] = df["parsed_response"].apply(is_valid_response)
df_clean = df[df["is_valid"]].drop(columns=["is_valid"]).reset_index(drop=True)

# 从干净记录重新抽样（覆盖之前的 df_sample）
df_sample = df_clean.sample(n=min(SAMPLE_SIZE, len(df_clean)), random_state=42).reset_index(drop=True)

print(f"✓ Records after quality filter : {len(df_clean)}")
print(f"✓ Dropped                      : {len(df) - len(df_clean)} records")
print(f"✓ Final sample size            : {len(df_sample)}")
print(f"\n--- Clean Sample Row ---")
print(f"Company  : {df_sample['company_name'].iloc[0]}")
print(f"Position : {df_sample['position_title'].iloc[0]}")
print(f"Parsed   : {json.dumps(df_sample['parsed_response'].iloc[0], indent=2)}")


In [ ]:
# ========== 前沿模型打分：build_prompt + score_with_frontier ==========

# OpenAI 客户端指向 OpenRouter
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

def build_prompt(row):
    """构造英文评分 prompt（发给模型的内容勿改译）。"""
    parsed = row["parsed_response"]
    return f"""You are a B2B SaaS sales intelligence tool.

Analyze the job posting below and score how likely this company is to be
in-market for a B2B SaaS product RIGHT NOW, based on their hiring signals.

Company       : {row['company_name']}
Position      : {row['position_title']}
Responsibilities: {parsed.get('Core Responsibilities', 'N/A')}
Required Skills : {parsed.get('Required Skills', 'N/A')}
Experience Level: {parsed.get('Experience Level', 'N/A')}

Respond ONLY with a JSON object in this exact format:
{{
  "score": <integer 0-100>,
  "tier": "<Hot|Warm|Cold>",
  "reasoning": "<one sentence explanation>"
}}"""

def score_with_frontier(row):
    """调用前沿模型打分；失败返回 score=0 / tier=Cold / error 信息。"""
    try:
        response = client.chat.completions.create(
            model=FRONTIER_MODEL,
            messages=[{"role": "user", "content": build_prompt(row)}],
            temperature=0.1,
        )
        raw = response.choices[0].message.content.strip()
        # 去掉可能的 ```json 代码围栏
        raw = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(raw)
        return {
            "score":     int(result.get("score", 0)),
            "tier":      result.get("tier", "Cold"),
            "reasoning": result.get("reasoning", ""),
            "error":     None
        }
    except Exception as e:
        return {"score": 0, "tier": "Cold", "reasoning": "", "error": str(e)}

# 先对一条做健全性检查，确认 API 可达
test_result = score_with_frontier(df_sample.iloc[0])
print(f"✓ Frontier model reachable")
print(f"  Company  : {df_sample.iloc[0]['company_name']}")
print(f"  Position : {df_sample.iloc[0]['position_title']}")
print(f"  Score    : {test_result['score']}")
print(f"  Tier     : {test_result['tier']}")
print(f"  Reasoning: {test_result['reasoning']}")
print(f"  Error    : {test_result['error']}")


In [ ]:
# ========== 对整份样本跑前沿模型并落盘 ==========

results = []

# 逐行打分（同步循环，便于进度打印与限速）
for i, row in df_sample.iterrows():
    result = score_with_frontier(row)
    results.append({
        "company_name":   row["company_name"],
        "position_title": row["position_title"],
        "score":          result["score"],
        "tier":           result["tier"],
        "reasoning":      result["reasoning"],
        "error":          result["error"]
    })

    # 每 20 条打印一次进度
    if (len(results)) % 20 == 0:
        print(f"  Processed {len(results)}/200...")

    # 轻微 sleep，降低触发 rate limit 的概率
    time.sleep(0.5)  # avoid rate limiting

# 结果转 DataFrame 并写成 JSON（orient=records）
df_results = pd.DataFrame(results)
df_results.to_json(os.path.join(OUTPUT_DIR, RESULTS_FILE), orient="records", indent=2)

# 汇总错误数、tier 分布、分数统计
errors = df_results["error"].notnull().sum()
print(f"\n✓ Scoring complete")
print(f"  Total scored : {len(df_results)}")
print(f"  Errors       : {errors}")
print(f"\n--- Tier Distribution ---")
print(df_results["tier"].value_counts().to_string())
print(f"\n--- Score Stats ---")
print(df_results["score"].describe().round(2).to_string())


In [ ]:
# ========== 可视化前沿模型：tier 条形图 / 分数直方图 / 箱线图 ==========

# 1×3 子图画板
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Frontier Model (GPT-4o-mini) — Scoring Results", fontsize=14, fontweight="bold")

# 1) 层级分布条形图（固定 Hot→Warm→Cold 顺序）
tier_counts = df_results["tier"].value_counts().reindex(["Hot", "Warm", "Cold"])
colors = ["#e74c3c", "#f39c12", "#3498db"]
axes[0].bar(tier_counts.index, tier_counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Tier Distribution")
axes[0].set_xlabel("Tier")
axes[0].set_ylabel("Count")
for i, v in enumerate(tier_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

# 2) 分数直方图 + 均值竖线
axes[1].hist(df_results["score"], bins=20, color="#2ecc71", edgecolor="black", linewidth=0.5)
axes[1].set_title("Score Distribution")
axes[1].set_xlabel("Score (0-100)")
axes[1].set_ylabel("Frequency")
axes[1].axvline(df_results["score"].mean(), color="red", linestyle="--", label=f'Mean: {df_results["score"].mean():.1f}')
axes[1].legend()

# 3) 按 tier 的分数箱线图
tier_order = ["Hot", "Warm", "Cold"]
tier_data = [df_results[df_results["tier"] == t]["score"].values for t in tier_order]
bp = axes[2].boxplot(tier_data, labels=tier_order, patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title("Score Range by Tier")
axes[2].set_xlabel("Tier")
axes[2].set_ylabel("Score")

# 收紧布局、保存到 Drive、展示
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "frontier_results.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\n--- Score by Tier ---")
print(df_results.groupby("tier")["score"].describe().round(2).to_string())
print(f"\n✓ Plot saved to {OUTPUT_DIR}/frontier_results.png")


In [ ]:
# ========== 把前沿打分结果做成 Llama 指令微调数据 ==========

def build_training_example(row, result):
    """一条训练样本：prompt + JSON completion，包进 Llama 特殊 token。"""
    prompt = build_prompt(row)
    completion = json.dumps({
        "score":     result["score"],
        "tier":      result["tier"],
        "reasoning": result["reasoning"]
    })
    # Llama-3 Instruct 聊天格式（begin_of_text / header / eot）
    return {
        "text": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{prompt}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n{completion}<|eot_id|>"
    }

# 跳过打分出错的记录，其余全部做成训练例
training_data = [
    build_training_example(df_sample.iloc[i], results[i])
    for i in range(len(df_sample))
    if results[i]["error"] is None
]

# 80/20 切分 train / eval
split = int(len(training_data) * 0.8)
train_data = training_data[:split]
eval_data  = training_data[split:]

# 路径：落盘到 Drive outputs
train_path = os.path.join(OUTPUT_DIR, "train.json")
eval_path  = os.path.join(OUTPUT_DIR, "eval.json")

with open(train_path, "w") as f:
    json.dump(train_data, f, indent=2)

with open(eval_path, "w") as f:
    json.dump(eval_data, f, indent=2)

print(f"✓ Training examples : {len(train_data)}")
print(f"✓ Eval examples     : {len(eval_data)}")
print(f"\n--- Sample Training Record ---")
print(training_data[0]["text"][:500] + "...")


In [ ]:
# ========== 再次 HF 登录（微调前确保凭证有效） ==========

login(token=HF_TOKEN, add_to_git_credential=False)


In [ ]:
# ========== 加载基座模型：4-bit NF4 量化（QLoRA 准备） ==========

# BitsAndBytes 4-bit 配置：省显存，适配 Colab 单卡
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 加载与模型匹配的分词器
tokenizer = AutoTokenizer.from_pretrained(
    OSS_MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)
# Llama 常缺 pad_token：用 eos 顶上；右侧 padding 利于因果 LM
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 按 4-bit 配置加载因果语言模型；device_map=auto 自动切设备
model = AutoModelForCausalLM.from_pretrained(
    OSS_MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"✓ Model loaded     : {OSS_MODEL_NAME}")
print(f"✓ Quantization     : 4-bit NF4")
print(f"✓ Compute dtype    : bfloat16")
print(f"✓ Device map       : auto")

# 看当前已占用 GPU 显存
mem = torch.cuda.memory_allocated() / 1024**3
print(f"✓ GPU memory used  : {mem:.2f} GB")


In [ ]:
# ========== 挂上 LoRA 适配器（只训少量参数） ==========

# k-bit 训练前处理：冻结基座、开梯度检查点等
model = prepare_model_for_kbit_training(model)

# LoRA：在注意力投影上插低秩矩阵
lora_config = LoraConfig(
    r=16,                     # rank：容量与显存的折中
    lora_alpha=32,            # 缩放因子（scaling）
    target_modules=[          # 要适配的注意力线性层
        "q_proj", "k_proj",
        "v_proj", "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 把 LoRA 接到模型上 → PeftModel
model = get_peft_model(model, lora_config)

# 打印可训练参数占比（应远小于总参数）
trainable, total = model.get_nb_trainable_parameters()
print(f"✓ LoRA adapters attached")
print(f"  Trainable params : {trainable:,}")
print(f"  Total params     : {total:,}")
print(f"  Trainable %      : {100 * trainable / total:.4f}%")


In [ ]:
# ========== 读 JSON → HF Dataset → tokenize ==========

from datasets import Dataset

# 从 Drive 读回 train/eval
with open(train_path, "r") as f:
    train_records = json.load(f)

with open(eval_path, "r") as f:
    eval_records = json.load(f)

# list[dict] → HuggingFace Dataset
train_dataset = Dataset.from_list(train_records)
eval_dataset  = Dataset.from_list(eval_records)

# 最大序列长度（再长就截断）
MAX_SEQ_LENGTH = 512

def tokenize(example):
    """把 text 字段编成 input_ids / attention_mask。"""
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

# batched map：批量分词更快
train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset  = eval_dataset.map(tokenize, batched=True)

print(f"✓ Train dataset : {len(train_dataset)} examples")
print(f"✓ Eval dataset  : {len(eval_dataset)} examples")
print(f"✓ Max seq length: {MAX_SEQ_LENGTH}")
print(f"\n--- Token length sample ---")
sample_len = len(train_dataset[0]["input_ids"])
print(f"  First example token length: {sample_len}")


In [ ]:
# ========== TrainingArguments + SFTTrainer：开训并保存 ==========

# 训练超参：小 batch + 梯度累积 ≈ 更大有效 batch；bf16 加速
training_args = TrainingArguments(
    output_dir=FINE_TUNED_MODEL_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

# SFTTrainer：监督微调封装（processing_class=tokenizer）
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("✓ Trainer configured")
print(f"  Epochs           : {training_args.num_train_epochs}")
print(f"  Batch size       : {training_args.per_device_train_batch_size}")
print(f"  Grad accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Learning rate    : {training_args.learning_rate}")
print(f"\nStarting training...")

# 真正开训（耗时，取决于 GPU）
train_result = trainer.train()

print(f"\n✓ Training complete")
print(f"  Runtime         : {train_result.metrics['train_runtime']:.0f}s")
print(f"  Train loss      : {train_result.metrics['train_loss']:.4f}")

# 保存适配器权重 + tokenizer 到 Drive
trainer.save_model(FINE_TUNED_MODEL_DIR)
tokenizer.save_pretrained(FINE_TUNED_MODEL_DIR)
print(f"✓ Model saved to  : {FINE_TUNED_MODEL_DIR}")


In [ ]:
# ========== 清显存 → 加载微调模型 → 本地推理打分 ==========

import gc
from peft import PeftModel

# 释放训练残留，腾出显存给推理
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"✓ GPU cleared : {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated")

# 重新加载 4-bit 基座
ft_model = AutoModelForCausalLM.from_pretrained(
    OSS_MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 挂上刚训好的 LoRA；切 eval 模式
ft_model = PeftModel.from_pretrained(ft_model, FINE_TUNED_MODEL_DIR)
ft_model.eval()
print(f"✓ Fine-tuned model loaded")
print(f"  GPU memory allocated : {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

def extract_json(text):
    """用正则抠出文本里第一个 {...} JSON 对象。"""
    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        return match.group(0)
    return None

def score_with_finetuned(row):
    """本地 generate：同一套 prompt，解析模型吐出的 JSON。"""
    prompt = build_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # 只 decode 新生成部分（去掉 prompt token）
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        clean    = generated.replace("```json", "").replace("```", "").strip()
        json_str = extract_json(clean)
        if not json_str:
            raise ValueError("No JSON object found in output")
        result = json.loads(json_str)
        return {
            "score":     int(result.get("score", 0)),
            "tier":      result.get("tier", "Cold"),
            "reasoning": result.get("reasoning", ""),
            "error":     None
        }
    except Exception as e:
        return {"score": 0, "tier": "Cold", "reasoning": "", "error": str(e)}

# 单条健全性检查
test_ft = score_with_finetuned(df_sample.iloc[0])
print(f"\n✓ Inference working")
print(f"  Company  : {df_sample.iloc[0]['company_name']}")
print(f"  Position : {df_sample.iloc[0]['position_title']}")
print(f"  Score    : {test_ft['score']}")
print(f"  Tier     : {test_ft['tier']}")
print(f"  Reasoning: {test_ft['reasoning']}")
print(f"  Error    : {test_ft['error']}")


In [ ]:
# ========== 整样本本地推理：分批打分并中途落盘 ==========

BATCH_SIZE = 40
ft_results = []
batch_num  = 0

# 按 BATCH_SIZE 切块，降低中断时的损失
for start in range(0, len(df_sample), BATCH_SIZE):
    batch     = df_sample.iloc[start:start + BATCH_SIZE]
    batch_num += 1
    print(f"\n  Processing batch {batch_num} ({start+1}-{min(start+BATCH_SIZE, len(df_sample))})...")

    for i, row in batch.iterrows():
        result = score_with_finetuned(row)
        ft_results.append({
            "company_name":   row["company_name"],
            "position_title": row["position_title"],
            "score":          result["score"],
            "tier":           result["tier"],
            "reasoning":      result["reasoning"],
            "error":          result["error"]
        })

    # 每批结束后立刻写 JSON，方便断点续跑
    df_ft_results = pd.DataFrame(ft_results)
    df_ft_results.to_json(
        os.path.join(OUTPUT_DIR, "finetuned_results.json"),
        orient="records", indent=2
    )
    print(f"  ✓ Batch {batch_num} saved — {len(ft_results)} total so far")

# 汇总错误与分布
errors = df_ft_results["error"].notnull().sum()
print(f"\n✓ Scoring complete")
print(f"  Total scored : {len(df_ft_results)}")
print(f"  Errors       : {errors}")
print(f"\n--- Tier Distribution ---")
print(df_ft_results["tier"].value_counts().to_string())
print(f"\n--- Score Stats ---")
print(df_ft_results["score"].describe().round(2).to_string())


In [ ]:
# ========== 最终对比：一致性 / 相关 / 多样性 + 对比图 + 结论 ==========

# 读回两套结果
df_frontier  = pd.read_json(os.path.join(OUTPUT_DIR, RESULTS_FILE))
df_finetuned = pd.read_json(os.path.join(OUTPUT_DIR, "finetuned_results.json"))

# 对齐前 100 条，保证逐行可比
df_frontier  = df_frontier.iloc[:100].reset_index(drop=True)
df_finetuned = df_finetuned.iloc[:100].reset_index(drop=True)

# 1) tier 一致率
agreement      = (df_frontier["tier"] == df_finetuned["tier"]).sum()
agreement_rate = agreement / len(df_frontier) * 100

# 2) 分数皮尔逊相关
correlation = df_frontier["score"].corr(df_finetuned["score"])

# 3) 分数标准差：衡量打分离散程度（多样性）
frontier_std  = df_frontier["score"].std()
finetuned_std = df_finetuned["score"].std()

# 4) 打印对比报告（表头英文保留，便于对照输出）
print("=" * 55)
print("        MODEL COMPARISON REPORT")
print("=" * 55)
print(f"  Records evaluated     : {len(df_frontier)}")
print(f"  Tier agreement rate   : {agreement_rate:.1f}%")
print(f"  Score correlation     : {correlation:.3f}")
print()
print(f"  {'Metric':<25} {'Frontier':>10} {'Fine-tuned':>10}")
print(f"  {'-'*45}")
print(f"  {'Mean score':<25} {df_frontier['score'].mean():>10.1f} {df_finetuned['score'].mean():>10.1f}")
print(f"  {'Std deviation':<25} {frontier_std:>10.2f} {finetuned_std:>10.2f}")
print(f"  {'Min score':<25} {df_frontier['score'].min():>10.1f} {df_finetuned['score'].min():>10.1f}")
print(f"  {'Max score':<25} {df_frontier['score'].max():>10.1f} {df_finetuned['score'].max():>10.1f}")
print()
print(f"  {'Tier':<10} {'Frontier':>10} {'Fine-tuned':>10}")
print(f"  {'-'*30}")
for tier in ["Hot", "Warm", "Cold"]:
    f_count  = (df_frontier["tier"]  == tier).sum()
    ft_count = (df_finetuned["tier"] == tier).sum()
    print(f"  {tier:<10} {f_count:>10} {ft_count:>10}")
print("=" * 55)

# 5) 三图对比：tier 柱状 / 分数叠直方 / 散点相关
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Frontier (GPT-4o-mini) vs Fine-Tuned (Llama 3.1 8B)", fontsize=14, fontweight="bold")

# 层级分布并排柱
tier_order   = ["Hot", "Warm", "Cold"]
frontier_counts  = [( df_frontier["tier"] == t).sum() for t in tier_order]
finetuned_counts = [(df_finetuned["tier"] == t).sum() for t in tier_order]
x = range(len(tier_order))
axes[0].bar([i - 0.2 for i in x], frontier_counts,  width=0.4, label="Frontier",  color="#3498db", edgecolor="black", linewidth=0.5)
axes[0].bar([i + 0.2 for i in x], finetuned_counts, width=0.4, label="Fine-tuned", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tier_order)
axes[0].set_title("Tier Distribution")
axes[0].set_ylabel("Count")
axes[0].legend()

# 分数分布叠加直方图
axes[1].hist(df_frontier["score"],  bins=20, alpha=0.6, label="Frontier",  color="#3498db", edgecolor="black", linewidth=0.5)
axes[1].hist(df_finetuned["score"], bins=20, alpha=0.6, label="Fine-tuned", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[1].set_title("Score Distribution")
axes[1].set_xlabel("Score (0-100)")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# 散点：x=前沿分，y=微调分；红虚线=完全一致
axes[2].scatter(df_frontier["score"], df_finetuned["score"], alpha=0.5, color="#2ecc71", edgecolor="black", linewidth=0.3)
axes[2].plot([0, 100], [0, 100], "r--", linewidth=1, label="Perfect agreement")
axes[2].set_title(f"Score Correlation (r={correlation:.2f})")
axes[2].set_xlabel("Frontier Score")
axes[2].set_ylabel("Fine-tuned Score")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

# 6) 文字结论：是否达到 70%+ tier 一致、谁的分数更分散
print("\n── VERDICT ──────────────────────────────────────────")
if agreement_rate >= 70:
    print(f"  ✅ Fine-tuned model MATCHES frontier tier accuracy")
    print(f"     ({agreement_rate:.1f}% tier agreement)")
else:
    print(f"    Fine-tuned model PARTIALLY matches frontier")
    print(f"     ({agreement_rate:.1f}% tier agreement — target: 70%+)")

if finetuned_std > frontier_std:
    print(f"   Fine-tuned model produces MORE score diversity")
    print(f"     (std {finetuned_std:.2f} vs frontier {frontier_std:.2f})")
else:
    print(f"    Frontier model produces more score diversity")
    print(f"     (std {frontier_std:.2f} vs fine-tuned {finetuned_std:.2f})")

print(f"   Score correlation : {correlation:.3f}")
print(f"   Training params   : 13.6M / 8B total (0.17%)")
print(f"   Inference cost    : $0 (local) vs API cost (frontier)")
print("─" * 55)
